In [ ]:
!pip install torch>=2.2.0 transformers>=4.51.0 datasets>=2.18.0 accelerate>=0.28.0 peft>=0.13.0 trl>=0.12.0 adapters
!pip install --upgrade torchao

In [ ]:

"""
Step 1 helper: load a small instruction dataset and format it for Qwen3.

We use 'yahma/alpaca-cleaned' — a free dataset on Hugging Face.
Each example becomes a simple user → assistant conversation.
"""

from datasets import Dataset, load_dataset


def load_data(max_samples: int = 500) -> tuple[Dataset, Dataset]:
    """
    Load Alpaca data and return (train, eval) splits.

    Args:
        max_samples: how many examples to use (keep small while learning)
    """
    raw = load_dataset("yahma/alpaca-cleaned", split="train")

    def to_messages(example):
        instruction = example["instruction"].strip()
        user_input = example.get("input", "").strip()
        answer = example["output"].strip()

        if user_input:
            question = f"{instruction}\n\nInput:\n{user_input}"
        else:
            question = instruction

        return {
            "messages": [
                {"role": "user", "content": question},
                {"role": "assistant", "content": answer},
            ]
        }

    dataset = raw.map(to_messages, remove_columns=raw.column_names)
    dataset = dataset.select(range(min(max_samples, len(dataset))))

    split = dataset.train_test_split(test_size=0.1, seed=42)
    return split["train"], split["test"]


def format_example(example, tokenizer) -> str:
    """Turn messages into the text Qwen3 expects."""
    return tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False,
    )f


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer


# ── CONFIG (change these if needed) ──────────────────────────────────────────
MODEL_NAME = "Qwen/Qwen3-4B"
OUTPUT_DIR = "./outputs/full"
MAX_SAMPLES = 500          # small set for learning; increase later
MAX_SEQ_LENGTH = 1024
EPOCHS = 1
BATCH_SIZE = 1
LEARNING_RATE = 2e-5
# ─────────────────────────────────────────────────────────────────────────────


def main():
    print("=" * 60)
    print("STEP 1: Load tokenizer")
    print("=" * 60)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token

    print("\n" + "=" * 60)
    print("STEP 2: Load model (ALL weights will be trained)")
    print("=" * 60)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
    )
    model.config.use_cache = False
    model.gradient_checkpointing_enable()

    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Total parameters:     {total:,}")
    print(f"  Trainable parameters: {trainable:,}  (100%)")

    print("\n" + "=" * 60)
    print("STEP 3: Load instruction dataset")
    print("=" * 60)
    train_data, eval_data = load_data(max_samples=MAX_SAMPLES)
    print(f"  Train examples: {len(train_data)}")
    print(f"  Eval examples:  {len(eval_data)}")

    def formatting_func(example):
        return format_example(example, tokenizer)

    print("\n" + "=" * 60)
    print("STEP 4: Set up trainer and train")
    print("=" * 60)
    training_args = SFTConfig(
        output_dir=OUTPUT_DIR,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=8,
        learning_rate=LEARNING_RATE,
        logging_steps=5,
        eval_strategy="steps",
        eval_steps=50,
        save_steps=100,
        bf16=True,
        max_length=MAX_SEQ_LENGTH,
        assistant_only_loss=True,
        report_to="none",
    )

    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=train_data,
        eval_dataset=eval_data,
        processing_class=tokenizer,
        formatting_func=formatting_func,
    )

    trainer.train()

    print("\n" + "=" * 60)
    print("STEP 5: Save model")
    print("=" * 60)
    trainer.save_model(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    print(f"  Saved to: {OUTPUT_DIR}")


if __name__ == "__main__":
    main()


STEP 1: Load tokenizer

STEP 2: Load model (ALL weights will be trained)


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

  Total parameters:     4,022,468,096
  Trainable parameters: 4,022,468,096  (100%)

STEP 3: Load instruction dataset
  Train examples: 450
  Eval examples:  50

STEP 4: Set up trainer and train


The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 11.81 MiB is free. Including non-PyTorch memory, this process has 14.55 GiB memory in use. Of the allocated memory 14.39 GiB is allocated by PyTorch, and 27.28 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
import torch

# Clear PyTorch's CUDA memory cache
torch.cuda.empty_cache()

In [ ]:
from __future__ import annotations

import os

import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM


class BottleneckAdapter(nn.Module):
    """Pfeiffer-style bottleneck: h' = h + Up(GELU(Down(h)))"""

    def __init__(self, hidden_size: int, adapter_size: int):
        super().__init__()
        self.down = nn.Linear(hidden_size, adapter_size, bias=False)
        self.up = nn.Linear(adapter_size, hidden_size, bias=False)
        self.act = nn.GELU()
        nn.init.normal_(self.down.weight, std=1e-3)
        nn.init.zeros_(self.up.weight)  # start as identity (no change)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.up(self.act(self.down(x)))


def _layer_dtype_device(layer) -> tuple[torch.dtype, torch.device]:
    """Match adapter dtype/device to the layer's existing weights."""
    param = next(layer.parameters())
    return param.dtype, param.device


def add_adapters_to_qwen3(model, adapter_size: int = 256) -> int:
    """
    Insert one bottleneck adapter after each layer's MLP.
    Returns the number of trainable parameters.
    """
    hidden_size = model.config.hidden_size

    for layer in model.model.layers:
        dtype, device = _layer_dtype_device(layer)
        adapter = BottleneckAdapter(hidden_size, adapter_size).to(device=device, dtype=dtype)
        layer.add_module("post_mlp_adapter", adapter)

        mlp = layer.mlp
        original_forward = mlp.forward

        def mlp_with_adapter(hidden_states, orig=original_forward, adap=adapter):
            return adap(orig(hidden_states))

        mlp.forward = mlp_with_adapter

    # Freeze base model, train adapters only
    for param in model.parameters():
        param.requires_grad = False
    for layer in model.model.layers:
        for param in layer.post_mlp_adapter.parameters():
            param.requires_grad = True

    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def save_adapters(model, output_dir: str) -> None:
    os.makedirs(output_dir, exist_ok=True)
    state = {
        "adapter_size": model.model.layers[0].post_mlp_adapter.down.out_features,
        "layers": {
            str(i): layer.post_mlp_adapter.state_dict()
            for i, layer in enumerate(model.model.layers)
        },
    }
    torch.save(state, os.path.join(output_dir, "bottleneck_adapters.pt"))


def load_adapters(model, adapter_dir: str) -> AutoModelForCausalLM:
    """Load base model and attach saved adapter weights."""
    path = os.path.join(adapter_dir, "bottleneck_adapters.pt")
    state = torch.load(path, map_location="cpu", weights_only=True)
    adapter_size = state["adapter_size"]

    add_adapters_to_qwen3(model, adapter_size=adapter_size)

    for i, layer in enumerate(model.model.layers):
        dtype, device = _layer_dtype_device(layer)
        layer.post_mlp_adapter.load_state_dict(state["layers"][str(i)])
        layer.post_mlp_adapter.to(device=device, dtype=dtype)

    return model


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer


# ── CONFIG ───────────────────────────────────────────────────────────────────
MODEL_NAME = "Qwen/Qwen3-4B"
OUTPUT_DIR = "./outputs/adapter"
MAX_SAMPLES = 500
MAX_SEQ_LENGTH = 1024
EPOCHS = 1
BATCH_SIZE = 1
LEARNING_RATE = 1e-4
ADAPTER_SIZE = 256   # bottleneck dimension (hidden_size / adapter_size = reduction factor)
# ─────────────────────────────────────────────────────────────────────────────


def main():
    print("=" * 60)
    print("STEP 1: Load tokenizer")
    print("=" * 60)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token

    print("\n" + "=" * 60)
    print("STEP 2: Load base model (will be FROZEN)")
    print("=" * 60)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
    )
    model.config.use_cache = False

    print("\n" + "=" * 60)
    print("STEP 3: Add bottleneck adapters after each layer's MLP")
    print("=" * 60)
    trainable = add_adapters_to_qwen3(model, adapter_size=ADAPTER_SIZE)
    total = sum(p.numel() for p in model.parameters())
    print(f"  Total parameters:     {total:,}")
    print(f"  Trainable parameters: {trainable:,}  ({100 * trainable / total:.2f}%)")

    model.gradient_checkpointing_enable()

    print("\n" + "=" * 60)
    print("STEP 4: Load instruction dataset")
    print("=" * 60)
    train_data, eval_data = load_data(max_samples=MAX_SAMPLES)
    print(f"  Train examples: {len(train_data)}")
    print(f"  Eval examples:  {len(eval_data)}")

    def formatting_func(example):
        return format_example(example, tokenizer)

    print("\n" + "=" * 60)
    print("STEP 5: Train adapters only")
    print("=" * 60)
    training_args = SFTConfig(
        output_dir=OUTPUT_DIR,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=8,
        learning_rate=LEARNING_RATE,
        logging_steps=5,
        eval_strategy="steps",
        eval_steps=50,
        save_steps=100,
        bf16=True,
        max_length=MAX_SEQ_LENGTH,
        assistant_only_loss=True,
        report_to="none",
    )

    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=train_data,
        eval_dataset=eval_data,
        processing_class=tokenizer,
        formatting_func=formatting_func,
    )

    trainer.train()

    print("\n" + "=" * 60)
    print("STEP 6: Save adapter weights (small file!)")
    print("=" * 60)
    save_adapters(model, OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    print(f"  Saved to: {OUTPUT_DIR}")


if __name__ == "__main__":
    main()


STEP 1: Load tokenizer

STEP 2: Load base model (will be FROZEN)


/usr/local/lib/python3.12/dist-packages/accelerate/utils/modeling.py:1583: UserWarning: Current model requires 128 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]


STEP 3: Add bottleneck adapters after each layer's MLP
  Total parameters:     4,069,654,016
  Trainable parameters: 47,185,920  (1.16%)

STEP 4: Load instruction dataset
  Train examples: 450
  Eval examples:  50

STEP 5: Train adapters only


Applying formatting function to train dataset:   0%|          | 0/450 [00:00<?, ? examples/s]

...

Tokenizing eval dataset:   0%|          | 0/50 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/50 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
50,0.996400,0.953151,0.973968,62947.000000,0.713006



STEP 6: Save adapter weights (small file!)
  Saved to: ./outputs/adapter


In [ ]:
import torch

# Clear PyTorch's CUDA memory cache
torch.cuda.empty_cache()

In [ ]:
!pip install --upgrade torchao
import torch
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer


# ── CONFIG ───────────────────────────────────────────────────────────────────
MODEL_NAME = "Qwen/Qwen3-4B"
OUTPUT_DIR = "./outputs/lora"
MAX_SAMPLES = 500
MAX_SEQ_LENGTH = 1024
EPOCHS = 1
BATCH_SIZE = 1
LEARNING_RATE = 2e-4
LORA_RANK = 16        # higher = more capacity, more memory
LORA_ALPHA = 32       # scaling factor (often 2x rank)
# ─────────────────────────────────────────────────────────────────────────────

# Layers where LoRA is attached (standard for Qwen/Llama models)
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]


def main():
    print("=" * 60)
    print("STEP 1: Load tokenizer")
    print("=" * 60)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token

    print("\n" + "=" * 60)
    print("STEP 2: Load base model (will be FROZEN)")
    print("=" * 60)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
    )
    model.config.use_cache = False

    print("\n" + "=" * 60)
    print("STEP 3: Add LoRA adapters")
    print("=" * 60)
    lora_config = LoraConfig(
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        lora_dropout=0.05,
        target_modules=LORA_TARGET_MODULES,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    model.gradient_checkpointing_enable()

    print("\n" + "=" * 60)
    print("STEP 4: Load instruction dataset")
    print("=" * 60)
    train_data, eval_data = load_data(max_samples=MAX_SAMPLES)
    print(f"  Train examples: {len(train_data)}")
    print(f"  Eval examples:  {len(eval_data)}")

    def formatting_func(example):
        return format_example(example, tokenizer)

    print("\n" + "=" * 60)
    print("STEP 5: Train LoRA weights only")
    print("=" * 60)
    training_args = SFTConfig(
        output_dir=OUTPUT_DIR,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=8,
        learning_rate=LEARNING_RATE,
        logging_steps=5,
        eval_strategy="steps",
        eval_steps=50,
        save_steps=100,
        bf16=True,
        max_length=MAX_SEQ_LENGTH,
        assistant_only_loss=True,
        report_to="none",
    )

    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=train_data,
        eval_dataset=eval_data,
        processing_class=tokenizer,
        formatting_func=formatting_func,
    )

    trainer.train()

    print("\n" + "=" * 60)
    print("STEP 6: Save LoRA adapter (small file!)")
    print("=" * 60)
    trainer.save_model(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    print(f"  Saved to: {OUTPUT_DIR}")


if __name__ == "__main__":
    main()

STEP 1: Load tokenizer


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Tokenizing eval dataset:   0%|          | 0/50 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/50 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
50,0.980700,0.939033,0.984210,62947.000000,0.712492



STEP 6: Save LoRA adapter (small file!)
  Saved to: ./outputs/lora
